# Phase 2 - Feature Engineering

**In:** `data/cleaned` (5.9M trips, 76 stations, 2019-2023)  
**Out:** `data/featured` - one row per `(station_id, start_date, hour)`

| Step | What |
| --- | --- |
| 1 | Aggregate trips to station-hour departures |
| 2 | Build the full grid and zero-fill |
| 3 | Join calendar features |
| 4 | Cyclical encoding |
| 5 | Write |

Features are **calendar-only**: no lags or rolling means. The app supplies
month/date/hour/station for a future date, so no trip history is available at
inference. Lags would predict better but could not be served.

## Step 1 - Aggregate to station-hour

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

EXPORT = Path("/opt/data/export")
ML = Path("/opt/data/ml")
FEATURED = ML / "featured"

CLEAN = f"read_parquet('{ML}/cleaned/**/*.parquet', hive_partitioning=1)"
POOL = f"read_parquet('{ML}/station_pool.parquet')"
DIM_DATE = f"read_parquet('{EXPORT}/dim_date/*.parquet')"

con = duckdb.connect()
con.execute("SET enable_progress_bar_print=false")

print(f"duckdb {duckdb.__version__} | pandas {pd.__version__}")

duckdb 1.5.5 | pandas 2.3.3


In [2]:
# Departures per station-hour. Only combinations that actually occurred - the
# zeros are added in step 2.
observed = con.execute(f"""
    SELECT station_id, start_date, hour, count(*) AS trip_count
    FROM {CLEAN}
    GROUP BY station_id, start_date, hour
""").df()

print(f"observed station-hours: {len(observed):,}")
print(f"trips accounted for:    {observed['trip_count'].sum():,}")
observed.head()

observed station-hours: 1,648,969
trips accounted for:    5,890,830


,station_id,start_date,hour,trip_count
0,7289,2019-03-28,18,2
1,7075,2019-03-01,7,1
2,7002,2019-03-28,19,3
3,7033,2019-03-28,19,4
4,7226,2019-03-28,19,2


## Step 2 - Grid and zero-fill

A station with no rentals at 3am is real zero demand and the model must learn
it. Training only on observed rows would drop ~50% of the panel and make the
model systematically over-predict.

Two exclusions:

- **Days with no system activity.** 30 of the 1,826 calendar days have zero
  trips network-wide (winter shutdowns). The grid uses observed dates only, so
  these never enter it.
- **Station lifespan.** Phase 1 already restricted the pool to stations
  operating across the full span, so no per-station bounds are needed here.

In [3]:
# Cross join over observed dates only, then left join the counts.
panel = con.execute(f"""
    WITH stations AS (SELECT station_id FROM {POOL}),
    dates AS (SELECT DISTINCT start_date FROM {CLEAN}),
    hours AS (SELECT unnest(generate_series(0, 23)) AS hour),
    grid AS (
        SELECT s.station_id, d.start_date, h.hour
        FROM stations s CROSS JOIN dates d CROSS JOIN hours h
    ),
    counts AS (
        SELECT station_id, start_date, hour, count(*) AS trip_count
        FROM {CLEAN} GROUP BY station_id, start_date, hour
    )
    SELECT g.station_id, g.start_date, g.hour,
        coalesce(c.trip_count, 0) AS trip_count
    FROM grid g
    LEFT JOIN counts c USING (station_id, start_date, hour)
""").df()

zeros = (panel["trip_count"] == 0).mean()
print(f"panel rows:  {len(panel):,}")
print(f"zero rows:   {zeros:.1%}")
print(f"trip total:  {panel['trip_count'].sum():,}")

# Zero-filling must not invent or lose trips.
assert panel["trip_count"].sum() == observed["trip_count"].sum(), "trip total changed"
assert not panel.duplicated(["station_id", "start_date", "hour"]).any(), "grain broken"

panel rows:  3,275,904
zero rows:   49.7%
trip total:  5,890,830


## Step 3 - Calendar features

`dim_date` already carries weekday, weekend, holiday, and season from the
warehouse build - no need to recompute them.

In [4]:
con.register("panel", panel)

featured = con.execute(f"""
    SELECT p.station_id, p.start_date, p.hour, p.trip_count,
        d.dim_date_year       AS year,
        d.dim_date_quarter    AS quarter,
        d.dim_date_month      AS month,
        d.dim_date_weekday    AS weekday,
        d.dim_date_week       AS week_of_year,
        d.dim_date_is_weekend AS is_weekend,
        d.dim_date_is_holiday AS is_holiday,
        d.dim_date_season     AS season
    FROM panel p
    JOIN {DIM_DATE} d ON d.dim_date_id = p.start_date
""").df()

# An inner join silently drops rows if dim_date has a gap.
assert len(featured) == len(panel), "calendar join dropped rows"

print(f"{len(featured):,} rows, {len(featured.columns)} columns")
featured.head()

3,275,904 rows, 12 columns


,station_id,start_date,hour,trip_count,year,quarter,month,weekday,week_of_year,is_weekend,is_holiday,season
0,7020,2019-10-15,0,3,2019,4,10,3,42,False,False,fall
1,7033,2019-10-15,0,1,2019,4,10,3,42,False,False,fall
2,7078,2019-10-15,0,1,2019,4,10,3,42,False,False,fall
3,7079,2019-10-15,0,1,2019,4,10,3,42,False,False,fall
4,7250,2019-10-15,0,2,2019,4,10,3,42,False,False,fall


## Step 4 - Cyclical encoding

Hour 23 and hour 0 are adjacent, but as raw integers they are 23 apart. Sin/cos
pairs put each cycle on a circle so the model sees that adjacency. Same for
weekday (Sun-Mon) and month (Dec-Jan).

The raw integers are kept as well - tree models split on them directly.

In [5]:
def cyclical(df, col, period):
    """Add sin/cos columns for a value that wraps every `period` units."""
    theta = 2 * np.pi * df[col] / period
    df[f"{col}_sin"] = np.sin(theta)
    df[f"{col}_cos"] = np.cos(theta)


cyclical(featured, "hour", 24)
cyclical(featured, "weekday", 7)
cyclical(featured, "month", 12)

featured["is_weekend"] = featured["is_weekend"].astype("int8")
featured["is_holiday"] = featured["is_holiday"].astype("int8")

# Adjacency check: the encoding is only useful if the wrap point is close.
h = featured.drop_duplicates("hour").set_index("hour")[["hour_sin", "hour_cos"]]
wrap = np.linalg.norm(h.loc[23].values - h.loc[0].values)
step = np.linalg.norm(h.loc[1].values - h.loc[0].values)
print(f"distance 23->0: {wrap:.4f} | 0->1: {step:.4f} (should match)")
assert np.isclose(wrap, step), "hour encoding does not wrap"

distance 23->0: 0.2611 | 0->1: 0.2611 (should match)


## Step 5 - Write

Partitioned by year so Phase 4 can take a chronological split by reading whole
partitions rather than filtering the lot.

In [6]:
FEATURED.mkdir(parents=True, exist_ok=True)
con.register("featured", featured)

con.execute(f"""
    COPY featured TO '{FEATURED}'
    (FORMAT parquet, PARTITION_BY year, OVERWRITE_OR_IGNORE 1)
""")

print(f"wrote -> {FEATURED}")

wrote -> /opt/data/ml/featured


In [7]:
# Read back and reconcile against what was written.
OUT = f"read_parquet('{FEATURED}/**/*.parquet', hive_partitioning=1)"

summary = con.execute(f"""
    SELECT year, count(*) AS n_rows,
        count(DISTINCT station_id) AS stations,
        sum(trip_count) AS trips,
        round(100.0 * count(*) FILTER (trip_count = 0) / count(*), 1) AS pct_zero
    FROM {OUT} GROUP BY year ORDER BY year
""").df()

print(summary.to_string(index=False))
print(f"\ntotal {summary['n_rows'].sum():,} rows | {summary['trips'].sum():,} trips")
assert summary["n_rows"].sum() == len(featured), "written rows do not match"
assert summary["trips"].sum() == observed["trip_count"].sum(), "trips lost in the write"

 year  n_rows  stations     trips  pct_zero
 2019  665760        76  919848.0      55.9
 2020  667584        76  936201.0      56.2
 2021  665760        76 1107431.0      49.9
 2022  611040        76 1291833.0      45.9
 2023  665760        76 1635517.0      40.1

total 3,275,904 rows | 5,890,830.0 trips


In [8]:
con.execute(f"DESCRIBE SELECT * FROM {OUT}").df()[["column_name", "column_type"]]

,column_name,column_type
0,station_id,INTEGER
1,start_date,TIMESTAMP
2,hour,BIGINT
3,trip_count,BIGINT
4,quarter,INTEGER
5,month,INTEGER
6,weekday,INTEGER
7,week_of_year,INTEGER
8,is_weekend,TINYINT
9,is_holiday,TINYINT
